In [ ]:
# STEP 1: Setup + load data

!pip install torch_geometric -q

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

# Upload earlyflag_graph_data.pt (or whichever real-data .pt file you're using)
from google.colab import files
uploaded = files.upload()  # select your .pt file when prompted

filename = list(uploaded.keys())[0]
data = torch.load(filename, weights_only=False)
data = data.to(device)

print(data)
print("Num nodes:", data.x.shape[0])
print("Num features:", data.x.shape[1])
print("Num classes:", int(data.y.max().item()) + 1)
print("Class counts:", torch.bincount(data.y))
print("Train/Val/Test sizes:", data.train_mask.sum().item(), data.val_mask.sum().item(), data.test_mask.sum().item())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00
Using device: cpu


Saving Real_school_data.pt to Real_school_data.pt
Data(x=[999, 36], edge_index=[2, 3387], edge_attr=[3387], y=[999], train_mask=[999], val_mask=[999], test_mask=[999])
Num nodes: 999
Num features: 36
Num classes: 3
Class counts: tensor([269, 450, 280])
Train/Val/Test sizes: 699 150 150


In [ ]:
# STEP 2: Plain (unmodified) GAT model

import torch.nn as nn
from torch_geometric.nn import GATConv

class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.5):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index, return_attention=False):
        x = F.dropout(x, p=self.dropout, training=self.training)

        if return_attention:
            x, (edge_index_1, alpha_1) = self.gat1(x, edge_index, return_attention_weights=True)
        else:
            x = self.gat1(x, edge_index)

        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        if return_attention:
            x, (edge_index_2, alpha_2) = self.gat2(x, edge_index, return_attention_weights=True)
            return x, (edge_index_1, alpha_1), (edge_index_2, alpha_2)
        else:
            x = self.gat2(x, edge_index)
            return x

in_channels = data.x.shape[1]      # 36
hidden_channels = 8
out_channels = 3                    # Low/Medium/High
heads = 4

plain_gat = GAT(in_channels, hidden_channels, out_channels, heads=heads).to(device)
print(plain_gat)

GAT(
  (gat1): GATConv(36, 8, heads=4)
  (gat2): GATConv(32, 3, heads=1)
)


In [ ]:
# STEP 3: Training loop + evaluation (macro-F1, per-class F1)

from sklearn.metrics import f1_score, classification_report
import copy

def train_model(model, data, epochs=200, lr=0.005, weight_decay=5e-4, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_f1 = 0
    best_state = None
    history = {'train_loss': [], 'val_macro_f1': []}

    for epoch in range(1, epochs + 1):
        # --- train step ---
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # --- validation step ---
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            val_true = data.y[data.val_mask].cpu().numpy()
            val_pred = pred[data.val_mask].cpu().numpy()
            val_macro_f1 = f1_score(val_true, val_pred, average='macro')

        history['train_loss'].append(loss.item())
        history['val_macro_f1'].append(val_macro_f1)

        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            best_state = copy.deepcopy(model.state_dict())

        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val Macro-F1: {val_macro_f1:.4f}")

    model.load_state_dict(best_state)
    print(f"\nBest Val Macro-F1: {best_val_f1:.4f}")
    return model, history


def evaluate_model(model, data, mask, label="Test"):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        y_true = data.y[mask].cpu().numpy()
        y_pred = pred[mask].cpu().numpy()

    macro_f1 = f1_score(y_true, y_pred, average='macro')
    per_class_f1 = f1_score(y_true, y_pred, average=None)

    print(f"--- {label} Results ---")
    print(f"Macro-F1: {macro_f1:.4f}")
    print(f"Per-class F1 (Low/Medium/High): {per_class_f1}")
    print("\n" + classification_report(y_true, y_pred, target_names=['Low', 'Medium', 'High'], zero_division=0))

    return macro_f1, per_class_f1

In [ ]:
# STEP 4: Train the plain (unmodified) GAT — this is your Section 2 baseline

torch.manual_seed(42)

plain_gat = GAT(in_channels, hidden_channels, out_channels, heads=heads).to(device)
plain_gat, plain_history = train_model(plain_gat, data, epochs=200, lr=0.005, weight_decay=5e-4)

print("\n" + "="*50)
plain_test_macro_f1, plain_test_per_class_f1 = evaluate_model(plain_gat, data, data.test_mask, label="Plain GAT — Test")

Epoch 020 | Loss: 1.3186 | Val Macro-F1: 0.2896
Epoch 040 | Loss: 1.1023 | Val Macro-F1: 0.3226
Epoch 060 | Loss: 1.0697 | Val Macro-F1: 0.3226
Epoch 080 | Loss: 1.0530 | Val Macro-F1: 0.2969
Epoch 100 | Loss: 1.0549 | Val Macro-F1: 0.2969
Epoch 120 | Loss: 1.0272 | Val Macro-F1: 0.2718
Epoch 140 | Loss: 1.0209 | Val Macro-F1: 0.2969
Epoch 160 | Loss: 1.0251 | Val Macro-F1: 0.2846
Epoch 180 | Loss: 1.0407 | Val Macro-F1: 0.2929
Epoch 200 | Loss: 1.0057 | Val Macro-F1: 0.2929

Best Val Macro-F1: 0.3511

--- Plain GAT — Test Results ---
Macro-F1: 0.3272
Per-class F1 (Low/Medium/High): [0.45454545 0.52702703 0.        ]

              precision    recall  f1-score   support

         Low       0.36      0.61      0.45        41
      Medium       0.48      0.58      0.53        67
        High       0.00      0.00      0.00        42

    accuracy                           0.43       150
   macro avg       0.28      0.40      0.33       150
weighted avg       0.31      0.43      0.36     

In [ ]:
# STEP 5: Gated GAT — the engineered model from Section 3

class GatedGAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.5):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)

        # Projects raw input features into the same space as the graph representation
        self.own_proj = nn.Linear(in_channels, out_channels)

        # Gate: takes [raw features, graph representation] -> scalar per node
        self.gate = nn.Linear(in_channels + out_channels, 1)

        self.dropout = dropout

    def forward(self, x, edge_index, return_gate=False):
        x_in = x  # keep raw input features for the gate and own_proj

        h = F.dropout(x, p=self.dropout, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        graph_repr = self.gat2(h, edge_index)  # [N, out_channels]

        own_repr = self.own_proj(x_in)  # [N, out_channels]

        gate_input = torch.cat([x_in, graph_repr], dim=1)
        gate_val = torch.sigmoid(self.gate(gate_input))  # [N, 1]

        out = gate_val * graph_repr + (1 - gate_val) * own_repr

        if return_gate:
            return out, gate_val
        return out

gated_gat = GatedGAT(in_channels, hidden_channels, out_channels, heads=heads).to(device)
print(gated_gat)

GatedGAT(
  (gat1): GATConv(36, 8, heads=4)
  (gat2): GATConv(32, 3, heads=1)
  (own_proj): Linear(in_features=36, out_features=3, bias=True)
  (gate): Linear(in_features=39, out_features=1, bias=True)
)


In [ ]:
# STEP 6: Train the gated GAT

torch.manual_seed(42)

gated_gat = GatedGAT(in_channels, hidden_channels, out_channels, heads=heads).to(device)
gated_gat, gated_history = train_model(gated_gat, data, epochs=200, lr=0.005, weight_decay=5e-4)

print("\n" + "="*50)
gated_test_macro_f1, gated_test_per_class_f1 = evaluate_model(gated_gat, data, data.test_mask, label="Gated GAT — Test")

print("\n" + "="*50)
print("COMPARISON")
print(f"Plain GAT  Macro-F1: {plain_test_macro_f1:.4f}")
print(f"Gated GAT  Macro-F1: {gated_test_macro_f1:.4f}")
print(f"Change: {gated_test_macro_f1 - plain_test_macro_f1:+.4f}")

Epoch 020 | Loss: 0.9567 | Val Macro-F1: 0.4968
Epoch 040 | Loss: 0.8900 | Val Macro-F1: 0.4865
Epoch 060 | Loss: 0.8488 | Val Macro-F1: 0.5760
Epoch 080 | Loss: 0.8273 | Val Macro-F1: 0.5956
Epoch 100 | Loss: 0.8113 | Val Macro-F1: 0.5848
Epoch 120 | Loss: 0.8012 | Val Macro-F1: 0.5939
Epoch 140 | Loss: 0.7931 | Val Macro-F1: 0.6027
Epoch 160 | Loss: 0.7868 | Val Macro-F1: 0.6142
Epoch 180 | Loss: 0.7825 | Val Macro-F1: 0.6142
Epoch 200 | Loss: 0.7767 | Val Macro-F1: 0.6085

Best Val Macro-F1: 0.6142

--- Gated GAT — Test Results ---
Macro-F1: 0.5531
Per-class F1 (Low/Medium/High): [0.61904762 0.54676259 0.49350649]

              precision    recall  f1-score   support

         Low       0.60      0.63      0.62        41
      Medium       0.53      0.57      0.55        67
        High       0.54      0.45      0.49        42

    accuracy                           0.55       150
   macro avg       0.56      0.55      0.55       150
weighted avg       0.55      0.55      0.55     

In [ ]:
# STEP 7: Inspect learned gate values

gated_gat.eval()
with torch.no_grad():
    out, gate_vals = gated_gat(data.x, data.edge_index, return_gate=True)
    gate_vals = gate_vals.squeeze().cpu().numpy()
    y_true = data.y.cpu().numpy()

print(f"Overall mean gate value: {gate_vals.mean():.4f}")
print(f"Overall std gate value:  {gate_vals.std():.4f}")
print(f"Min / Max gate value:    {gate_vals.min():.4f} / {gate_vals.max():.4f}")

print("\nMean gate value by true class:")
for cls, name in zip([0, 1, 2], ['Low', 'Medium', 'High']):
    mask = y_true == cls
    print(f"  {name:8s}: {gate_vals[mask].mean():.4f}  (n={mask.sum()})")

print("\nMean gate value on test set only:")
test_mask_np = data.test_mask.cpu().numpy()
print(f"  Test set: {gate_vals[test_mask_np].mean():.4f}")

# Distribution shape
import numpy as np
print("\nGate value percentiles (test set):")
for p in [10, 25, 50, 75, 90]:
    print(f"  {p}th percentile: {np.percentile(gate_vals[test_mask_np], p):.4f}")

Overall mean gate value: 0.0049
Overall std gate value:  0.0057
Min / Max gate value:    0.0003 / 0.0431

Mean gate value by true class:
  Low     : 0.0043  (n=269)
  Medium  : 0.0050  (n=450)
  High    : 0.0055  (n=280)

Mean gate value on test set only:
  Test set: 0.0054

Gate value percentiles (test set):
  10th percentile: 0.0013
  25th percentile: 0.0018
  50th percentile: 0.0029
  75th percentile: 0.0072
  90th percentile: 0.0116


In [ ]:
# STEP 8: MLP-only baseline (no graph) — sanity check for gate interpretation

class MLP(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(in_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index=None):  # edge_index accepted but unused, so train_model/evaluate_model work unchanged
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        return x

torch.manual_seed(42)

mlp_model = MLP(in_channels, hidden_channels=32, out_channels=out_channels).to(device)
mlp_model, mlp_history = train_model(mlp_model, data, epochs=200, lr=0.005, weight_decay=5e-4)

print("\n" + "="*50)
mlp_test_macro_f1, mlp_test_per_class_f1 = evaluate_model(mlp_model, data, data.test_mask, label="MLP-only (no graph) — Test")

print("\n" + "="*50)
print("FULL COMPARISON")
print(f"Plain GAT       Macro-F1: {plain_test_macro_f1:.4f}")
print(f"Gated GAT       Macro-F1: {gated_test_macro_f1:.4f}")
print(f"MLP-only (no graph) Macro-F1: {mlp_test_macro_f1:.4f}")

Epoch 020 | Loss: 0.9575 | Val Macro-F1: 0.4805
Epoch 040 | Loss: 0.9232 | Val Macro-F1: 0.5013
Epoch 060 | Loss: 0.9259 | Val Macro-F1: 0.4910
Epoch 080 | Loss: 0.9036 | Val Macro-F1: 0.4952
Epoch 100 | Loss: 0.9147 | Val Macro-F1: 0.5084
Epoch 120 | Loss: 0.9258 | Val Macro-F1: 0.5019
Epoch 140 | Loss: 0.8878 | Val Macro-F1: 0.5226
Epoch 160 | Loss: 0.9221 | Val Macro-F1: 0.4846
Epoch 180 | Loss: 0.9018 | Val Macro-F1: 0.5132
Epoch 200 | Loss: 0.8921 | Val Macro-F1: 0.5099

Best Val Macro-F1: 0.5488

--- MLP-only (no graph) — Test Results ---
Macro-F1: 0.4915
Per-class F1 (Low/Medium/High): [0.55       0.56962025 0.35483871]

              precision    recall  f1-score   support

         Low       0.56      0.54      0.55        41
      Medium       0.49      0.67      0.57        67
        High       0.55      0.26      0.35        42

    accuracy                           0.52       150
   macro avg       0.54      0.49      0.49       150
weighted avg       0.53      0.52     

In [ ]:
# STEP 9: Bare linear-only baseline (matches own_proj exactly — the true counterfactual)

class LinearOnly(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.lin = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index=None):
        return self.lin(x)

torch.manual_seed(42)

linear_model = LinearOnly(in_channels, out_channels).to(device)
linear_model, linear_history = train_model(linear_model, data, epochs=200, lr=0.005, weight_decay=5e-4)

print("\n" + "="*50)
linear_test_macro_f1, linear_test_per_class_f1 = evaluate_model(linear_model, data, data.test_mask, label="Linear-only (matches own_proj) — Test")

print("\n" + "="*50)
print("FULL COMPARISON")
print(f"Plain GAT             Macro-F1: {plain_test_macro_f1:.4f}")
print(f"Gated GAT             Macro-F1: {gated_test_macro_f1:.4f}")
print(f"MLP-only (no graph)   Macro-F1: {mlp_test_macro_f1:.4f}")
print(f"Linear-only (own_proj match) Macro-F1: {linear_test_macro_f1:.4f}")

Epoch 020 | Loss: 0.9450 | Val Macro-F1: 0.3769
Epoch 040 | Loss: 0.8692 | Val Macro-F1: 0.5827
Epoch 060 | Loss: 0.8350 | Val Macro-F1: 0.5793
Epoch 080 | Loss: 0.8166 | Val Macro-F1: 0.6123
Epoch 100 | Loss: 0.8053 | Val Macro-F1: 0.6098
Epoch 120 | Loss: 0.7975 | Val Macro-F1: 0.5972
Epoch 140 | Loss: 0.7919 | Val Macro-F1: 0.5972
Epoch 160 | Loss: 0.7874 | Val Macro-F1: 0.6052
Epoch 180 | Loss: 0.7838 | Val Macro-F1: 0.5972
Epoch 200 | Loss: 0.7806 | Val Macro-F1: 0.5972

Best Val Macro-F1: 0.6182

--- Linear-only (matches own_proj) — Test Results ---
Macro-F1: 0.4858
Per-class F1 (Low/Medium/High): [0.53846154 0.51351351 0.40540541]

              precision    recall  f1-score   support

         Low       0.57      0.51      0.54        41
      Medium       0.47      0.57      0.51        67
        High       0.47      0.36      0.41        42

    accuracy                           0.49       150
   macro avg       0.50      0.48      0.49       150
weighted avg       0.50    

In [ ]:
# STEP 10: Diagnostic - does y actually correlate with the features that should have built it?

import pandas as pd
from scipy.stats import spearmanr

x_np = data.x.cpu().numpy()
y_np = data.y.cpu().numpy()

# 1) Rank all 36 features by how strongly they correlate with y (ordinal: 0=Low,1=Medium,2=High)
correlations = []
for i in range(x_np.shape[1]):
    corr, pval = spearmanr(x_np[:, i], y_np)
    correlations.append((i, corr, pval))

corr_df = pd.DataFrame(correlations, columns=['feature_idx', 'spearman_corr', 'p_value'])
corr_df['abs_corr'] = corr_df['spearman_corr'].abs()
corr_df = corr_df.sort_values('abs_corr', ascending=False)

print("Top 10 features most correlated with y (label):")
print(corr_df.head(10).to_string(index=False))

print("\nBottom 10 features (weakest correlation with y):")
print(corr_df.tail(10).to_string(index=False))

# 2) Sanity check: if label was truly built from attendance/academic features with 0.65/0.35 weighting,
# we'd expect AT LEAST a few features with strong |correlation| (e.g. > 0.5), since percentile-ranking
# a weighted composite should preserve substantial rank correlation with its main inputs.
strong_features = corr_df[corr_df['abs_corr'] > 0.5]
print(f"\nNumber of features with |correlation| > 0.5: {len(strong_features)}")
print(f"Strongest single correlation found: {corr_df['abs_corr'].max():.4f}")

Top 10 features most correlated with y (label):
 feature_idx  spearman_corr      p_value  abs_corr
           2      -0.569872 4.290160e-87  0.569872
           1      -0.564988 2.547709e-85  0.564988
          35      -0.462812 3.547176e-54  0.462812
          34      -0.421910 2.171074e-44  0.421910
          14      -0.112551 3.648509e-04  0.112551
          15       0.086251 6.375749e-03  0.086251
          22      -0.070815 2.520454e-02  0.070815
          29       0.063666 4.424139e-02  0.063666
           0       0.061551 5.179490e-02  0.061551
          24       0.053748 8.952375e-02  0.053748

Bottom 10 features (weakest correlation with y):
 feature_idx  spearman_corr  p_value  abs_corr
          33       0.005888 0.852557  0.005888
           4       0.005301 0.867091  0.005301
          23       0.004742 0.881011  0.004742
          27       0.003841 0.903501  0.003841
           8       0.003795 0.904647  0.003795
          32      -0.003508 0.911818  0.003508
          31

In [ ]:
# STEP 11: Reconstruct composite score from top-correlated features and test against y

import numpy as np

# Candidate attendance features (strongest correlation, likely weighted 0.65)
attendance_feats = x_np[:, [1, 2]].mean(axis=1)

# Candidate academic features (moderate correlation, likely weighted 0.35)
academic_feats = x_np[:, [34, 35]].mean(axis=1)

# Reconstruct composite using documented weights
# Note: correlations are negative, so higher feature value = lower risk.
# We flip sign so higher composite = higher risk, matching y's direction (0=Low,...,2=High)
composite_score = -1 * (0.65 * attendance_feats + 0.35 * academic_feats)

# Rank into 3 tiers by percentile, matching the documented ~27/45/28 split
low_cut = np.percentile(composite_score, 27)
high_cut = np.percentile(composite_score, 72)

reconstructed_y = np.zeros_like(composite_score, dtype=int)
reconstructed_y[composite_score > high_cut] = 2  # High
reconstructed_y[(composite_score > low_cut) & (composite_score <= high_cut)] = 1  # Medium
# else stays 0 (Low)

# Compare to actual y
from sklearn.metrics import accuracy_score, f1_score
agreement = accuracy_score(y_np, reconstructed_y)
macro_f1_agreement = f1_score(y_np, reconstructed_y, average='macro')

print(f"Agreement between reconstructed label and actual y: {agreement:.4f}")
print(f"Macro-F1 between reconstructed label and actual y:  {macro_f1_agreement:.4f}")
print(f"\nReconstructed class counts: {np.bincount(reconstructed_y)}")
print(f"Actual y class counts:      {np.bincount(y_np)}")

Agreement between reconstructed label and actual y: 0.5495
Macro-F1 between reconstructed label and actual y:  0.5554

Reconstructed class counts: [270 449 280]
Actual y class counts:      [269 450 280]


In [ ]:
# STEP 12: Confirm collapse + check class balance (plain GAT, the one that actually collapsed)

print("Class distribution in this 999-node subset:")
print(pd.Series(data.y.cpu().numpy()).value_counts())
print(pd.Series(data.y.cpu().numpy()).value_counts(normalize=True).round(3))

plain_gat.eval()
with torch.no_grad():
    out = plain_gat(data.x, data.edge_index)
    preds = out[data.test_mask].argmax(dim=1)

print("\nPlain GAT predicted class distribution on test set:")
print(torch.bincount(preds, minlength=3))
print("Actual class distribution on test set:")
print(torch.bincount(data.y[data.test_mask], minlength=3))

Class distribution in this 999-node subset:
1    450
2    280
0    269
Name: count, dtype: int64
1    0.450
2    0.280
0    0.269
Name: proportion, dtype: float64

Plain GAT predicted class distribution on test set:
tensor([69, 81,  0])
Actual class distribution on test set:
tensor([41, 67, 42])


In [ ]:
# STEP 13: Class-weighted retraining — test on the model that actually collapsed (plain GAT)

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_labels = data.y[data.train_mask].cpu().numpy()
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_labels
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights (Low, Medium, High):", class_weights_tensor)

def train_model_weighted(model, data, epochs=200, lr=0.005, weight_decay=5e-4, verbose=True):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

    best_val_f1 = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out.argmax(dim=1)
            val_macro_f1 = f1_score(data.y[data.val_mask].cpu().numpy(), pred[data.val_mask].cpu().numpy(), average='macro')

        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            best_state = copy.deepcopy(model.state_dict())

        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f} | Val Macro-F1: {val_macro_f1:.4f}")

    model.load_state_dict(best_state)
    print(f"\nBest Val Macro-F1: {best_val_f1:.4f}")
    return model

torch.manual_seed(42)
plain_gat_weighted = GAT(in_channels, hidden_channels, out_channels, heads=heads).to(device)
plain_gat_weighted = train_model_weighted(plain_gat_weighted, data)

print("\n" + "="*50)
weighted_test_macro_f1, weighted_test_per_class_f1 = evaluate_model(plain_gat_weighted, data, data.test_mask, label="Plain GAT + class weights — Test")

print("\n" + "="*50)
print("COMPARISON")
print(f"Plain GAT (unweighted)     Macro-F1: {plain_test_macro_f1:.4f}, High F1: {plain_test_per_class_f1[2]:.4f}")
print(f"Plain GAT (class-weighted) Macro-F1: {weighted_test_macro_f1:.4f}, High F1: {weighted_test_per_class_f1[2]:.4f}")
print(f"Gated GAT (unweighted)     Macro-F1: {gated_test_macro_f1:.4f}, High F1: {gated_test_per_class_f1[2]:.4f}")

Class weights (Low, Medium, High): tensor([1.2394, 0.7397, 1.1888])
Epoch 020 | Loss: 1.3478 | Val Macro-F1: 0.4536
Epoch 040 | Loss: 1.1177 | Val Macro-F1: 0.4045
Epoch 060 | Loss: 1.0868 | Val Macro-F1: 0.4089
Epoch 080 | Loss: 1.0726 | Val Macro-F1: 0.4406
Epoch 100 | Loss: 1.0743 | Val Macro-F1: 0.4332
Epoch 120 | Loss: 1.0439 | Val Macro-F1: 0.4055
Epoch 140 | Loss: 1.0395 | Val Macro-F1: 0.4157
Epoch 160 | Loss: 1.0443 | Val Macro-F1: 0.4296
Epoch 180 | Loss: 1.0584 | Val Macro-F1: 0.4371
Epoch 200 | Loss: 1.0223 | Val Macro-F1: 0.4264

Best Val Macro-F1: 0.5230

--- Plain GAT + class weights — Test Results ---
Macro-F1: 0.4118
Per-class F1 (Low/Medium/High): [0.49019608 0.37931034 0.36585366]

              precision    recall  f1-score   support

         Low       0.41      0.61      0.49        41
      Medium       0.45      0.33      0.38        67
        High       0.38      0.36      0.37        42

    accuracy                           0.41       150
   macro avg      

In [ ]:
# STEP 14: Gated GAT + class weighting — does combining both help further?

torch.manual_seed(42)
gated_gat_weighted = GatedGAT(in_channels, hidden_channels, out_channels, heads=heads).to(device)
gated_gat_weighted = train_model_weighted(gated_gat_weighted, data)

print("\n" + "="*50)
gw_test_macro_f1, gw_test_per_class_f1 = evaluate_model(gated_gat_weighted, data, data.test_mask, label="Gated GAT + class weights — Test")

print("\n" + "="*50)
print("FULL COMPARISON")
print(f"Plain GAT (unweighted)      Macro-F1: {plain_test_macro_f1:.4f}, High F1: {plain_test_per_class_f1[2]:.4f}")
print(f"Plain GAT (weighted)        Macro-F1: {weighted_test_macro_f1:.4f}, High F1: {weighted_test_per_class_f1[2]:.4f}")
print(f"Gated GAT (unweighted)      Macro-F1: {gated_test_macro_f1:.4f}, High F1: {gated_test_per_class_f1[2]:.4f}")
print(f"Gated GAT (weighted)        Macro-F1: {gw_test_macro_f1:.4f}, High F1: {gw_test_per_class_f1[2]:.4f}")

Epoch 020 | Loss: 0.9580 | Val Macro-F1: 0.5642
Epoch 040 | Loss: 0.8838 | Val Macro-F1: 0.5308
Epoch 060 | Loss: 0.8374 | Val Macro-F1: 0.5185
Epoch 080 | Loss: 0.8119 | Val Macro-F1: 0.4972
Epoch 100 | Loss: 0.7939 | Val Macro-F1: 0.5016
Epoch 120 | Loss: 0.7817 | Val Macro-F1: 0.4942
Epoch 140 | Loss: 0.7722 | Val Macro-F1: 0.4998
Epoch 160 | Loss: 0.7649 | Val Macro-F1: 0.5083
Epoch 180 | Loss: 0.7599 | Val Macro-F1: 0.4915
Epoch 200 | Loss: 0.7534 | Val Macro-F1: 0.4969

Best Val Macro-F1: 0.5911

--- Gated GAT + class weights — Test Results ---
Macro-F1: 0.5115
Per-class F1 (Low/Medium/High): [0.67368421 0.38095238 0.48      ]

              precision    recall  f1-score   support

         Low       0.59      0.78      0.67        41
      Medium       0.53      0.30      0.38        67
        High       0.41      0.57      0.48        42

    accuracy                           0.51       150
   macro avg       0.51      0.55      0.51       150
weighted avg       0.51      0.5

In [8]:
!pip install torch_geometric -q

In [ ]:
"""
Group7 dropout prediction — model comparison script.
Run in Colab. Upload Real_school_data_full.pt to the Colab session first
(or mount Drive and adjust DATA_PATH), then run all cells top to bottom.

Models compared:
  - Logistic Regression   (non-graph baseline)
  - Random Forest         (non-graph baseline)
  - GraphSAGE             (graph, no attention)
  - GAT (unmodified)      (graph, attention)
  - Gated GAT             (our engineered intervention)
"""

# ---- 0. Setup ----------------------------------------------------------
# In Colab, run this once in its own cell first:
#   !pip install torch_geometric -q
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, SAGEConv
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import numpy as np

DATA_PATH = "Real_school_data_full.pt"  # adjust if uploaded elsewhere
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = torch.load(DATA_PATH, weights_only=False).to(device)

print(f"Device: {device}")
print(f"Nodes: {data.num_nodes} | Edges (incl. self-loops): {data.edge_index.shape[1]}")
print(f"Class counts: {torch.bincount(data.y).tolist()}")
print(f"Train/Val/Test sizes: {int(data.train_mask.sum())}/{int(data.val_mask.sum())}/{int(data.test_mask.sum())}")

# ---- 0b. Sanity check: candidate school column (col 1) ----------------
# Column 1 of x has exactly 3 unique values with group sizes matching a
# plausible 3-school split. CONFIRM with your teammates whether this is
# actually the school ID before trusting this check.
candidate_school_col = 1
grp = data.x[:, candidate_school_col].long()
src, dst = data.edge_index
non_self = src != dst
same_grp = (grp[src[non_self]] == grp[dst[non_self]])
print(f"\n[Sanity check] Using x[:, {candidate_school_col}] as candidate school/group id:")
print(f"  Group sizes: {torch.bincount(grp).tolist()}")
print(f"  Same-group edges: {same_grp.sum().item()} / {non_self.sum().item()} "
      f"({100*same_grp.float().mean().item():.1f}%)")
print("  ^ If this column IS school id and cross-group % is high, the graph")
print("    is not strictly per-school — confirm this is expected before trusting results.\n")

# ---- 1. Non-graph baselines --------------------------------------------
X = data.x.cpu().numpy()
y = data.y.cpu().numpy()
train_idx = data.train_mask.cpu().numpy()
test_idx = data.test_mask.cpu().numpy()

def eval_sklearn_baseline(model, name):
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict(X[test_idx])
    macro_f1 = f1_score(y[test_idx], preds, average="macro")
    per_class = f1_score(y[test_idx], preds, average=None)
    print(f"{name:22s} macro-F1={macro_f1:.3f}  per-class={np.round(per_class,3)}")
    return macro_f1

lr_f1 = eval_sklearn_baseline(LogisticRegression(max_iter=2000), "Logistic Regression")
rf_f1 = eval_sklearn_baseline(RandomForestClassifier(n_estimators=300, random_state=SEED), "Random Forest")

# ---- 2. Model definitions -----------------------------------------------
IN_DIM = data.x.shape[1]
HIDDEN = 16
OUT_DIM = int(data.y.max().item()) + 1

class GAT(nn.Module):
    """Unmodified 2-layer GAT — this is the Section 2 baseline being tested."""
    def __init__(self, in_dim, hidden, out_dim, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout, edge_dim=1)
        self.conv2 = GATConv(hidden * heads, out_dim, heads=1, concat=False, dropout=dropout, edge_dim=1)

    def forward(self, x, edge_index, edge_attr):
        e = edge_attr.unsqueeze(-1)
        x = F.elu(self.conv1(x, edge_index, e))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, e)
        return x

class SAGE(nn.Module):
    """GraphSAGE — same graph, mean aggregation, no attention. Isolates
    whether attention itself is the source of underperformance."""
    def __init__(self, in_dim, hidden, out_dim, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, out_dim)

    def forward(self, x, edge_index, edge_attr=None):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GatedGAT(nn.Module):
    """The engineered intervention from Section 3: a learned per-node
    sigmoid gate blending the GAT's graph-aggregated representation
    against a projection of the node's own raw features.
        out = gate * h_graph + (1 - gate) * h_own
    """
    def __init__(self, in_dim, hidden, out_dim, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout, edge_dim=1)
        self.conv2 = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout, edge_dim=1)
        self.own_proj = nn.Linear(in_dim, hidden)
        self.gate = nn.Linear(in_dim + hidden, 1)
        self.out = nn.Linear(hidden, out_dim)
        self.last_gate_values = None  # populated on forward, for post-hoc inspection

    def forward(self, x, edge_index, edge_attr):
        e = edge_attr.unsqueeze(-1)
        h_graph = F.elu(self.conv1(x, edge_index, e))
        h_graph = F.dropout(h_graph, p=self.dropout, training=self.training)
        h_graph = self.conv2(h_graph, edge_index, e)

        h_own = self.own_proj(x)
        g = torch.sigmoid(self.gate(torch.cat([x, h_graph], dim=-1)))
        self.last_gate_values = g.detach()

        h = g * h_graph + (1 - g) * h_own
        return self.out(h)

# ---- 3. Train / eval loops ----------------------------------------------
def train_gnn(model, data, epochs=200, lr=0.01, weight_decay=5e-4, patience=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_f1, best_state, stale = 0.0, None, 0
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index, data.edge_attr)
            val_preds = out[data.val_mask].argmax(dim=1)
            val_f1 = f1_score(data.y[data.val_mask].cpu(), val_preds.cpu(), average="macro")
        if val_f1 > best_val_f1:
            best_val_f1, best_state, stale = val_f1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= patience:
                break
    model.load_state_dict(best_state)
    return model

def eval_gnn(model, data, name):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index, data.edge_attr)
        preds = out[data.test_mask].argmax(dim=1).cpu()
        y_true = data.y[data.test_mask].cpu()
        macro_f1 = f1_score(y_true, preds, average="macro")
        per_class = f1_score(y_true, preds, average=None)
    print(f"{name:22s} macro-F1={macro_f1:.3f}  per-class={np.round(per_class,3)}")
    return macro_f1

print()
torch.manual_seed(SEED)
gat = GAT(IN_DIM, HIDDEN, OUT_DIM).to(device)
gat = train_gnn(gat, data)
gat_f1 = eval_gnn(gat, data, "GAT (unmodified)")

torch.manual_seed(SEED)
sage = SAGE(IN_DIM, HIDDEN, OUT_DIM).to(device)
sage = train_gnn(sage, data)
sage_f1 = eval_gnn(sage, data, "GraphSAGE")

torch.manual_seed(SEED)
gated = GatedGAT(IN_DIM, HIDDEN, OUT_DIM).to(device)
gated = train_gnn(gated, data)
gated_f1 = eval_gnn(gated, data, "Gated GAT")

# ---- 4. Gate value inspection --------------------------------------------
gated.eval()
with torch.no_grad():
    _ = gated(data.x, data.edge_index, data.edge_attr)
    gate_vals = gated.last_gate_values.cpu().numpy().flatten()
print(f"\nGate values -> mean={gate_vals.mean():.3f} std={gate_vals.std():.3f} "
      f"min={gate_vals.min():.3f} max={gate_vals.max():.3f}")
print("(mean close to 0 = leaning on own features; close to 1 = trusting the graph)")

# ---- 5. Summary -----------------------------------------------------------
print("\n=== SUMMARY (test set macro-F1) ===")
print(f"Logistic Regression: {lr_f1:.3f}")
print(f"Random Forest:       {rf_f1:.3f}")
print(f"GraphSAGE:           {sage_f1:.3f}")
print(f"GAT (unmodified):    {gat_f1:.3f}")
print(f"Gated GAT:           {gated_f1:.3f}")
print(f"\nDelta (Gated GAT - unmodified GAT): {gated_f1 - gat_f1:+.3f}")
print("For reference, the plan's n=611 Ayeduase-only run reported: LR=0.871, "
      "RF=0.877, SAGE=0.875, GAT=0.787, Gated GAT=0.808.")

Device: cpu
Nodes: 1000 | Edges (incl. self-loops): 3996
Class counts: [270, 449, 281]
Train/Val/Test sizes: 700/150/150

[Sanity check] Using x[:, 1] as candidate school/group id:
  Group sizes: [0, 128, 470, 402]
  Same-group edges: 1592 / 2996 (53.1%)
  ^ If this column IS school id and cross-group % is high, the graph
    is not strictly per-school — confirm this is expected before trusting results.

Logistic Regression    macro-F1=0.337  per-class=[0.254 0.518 0.239]
Random Forest          macro-F1=0.335  per-class=[0.246 0.566 0.194]

GAT (unmodified)       macro-F1=0.307  per-class=[0.338 0.512 0.073]
GraphSAGE              macro-F1=0.310  per-class=[0.348 0.544 0.039]
Gated GAT              macro-F1=0.315  per-class=[0.278 0.503 0.164]

Gate values -> mean=0.059 std=0.029 min=0.018 max=0.328
(mean close to 0 = leaning on own features; close to 1 = trusting the graph)

=== SUMMARY (test set macro-F1) ===
Logistic Regression: 0.337
Random Forest:       0.335
GraphSAGE:           

In [9]:
"""
Group7 dropout prediction — data diagnostic script.
Run this BEFORE re-running any model. Goal: figure out whether the low
macro-F1 across every model (including non-graph LR/RF) is due to
misaligned data, uninformative features, or something else.

Upload Real_school_data_full.pt to the Colab session first.
"""

import torch
import numpy as np
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score

DATA_PATH = "Real_school_data_full.pt"
data = torch.load(DATA_PATH, weights_only=False)

X = data.x.numpy()
y = data.y.numpy()

print("=== 0. Basic integrity checks ===")
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"NaNs in X: {np.isnan(X).sum()}, NaNs in y: {np.isnan(y.astype(float)).sum()}")
print(f"Duplicate rows in X: {X.shape[0] - len(np.unique(X, axis=0))}")
print(f"Class balance: {np.bincount(y)} (proportions: {np.round(np.bincount(y)/len(y),3)})")

# ---- 1. Trivial baseline: what does "predict majority class" get you? ----
print("\n=== 1. Trivial floor ===")
train_idx = data.train_mask.numpy()
test_idx = data.test_mask.numpy()
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X[train_idx], y[train_idx])
dummy_preds = dummy.predict(X[test_idx])
dummy_f1 = f1_score(y[test_idx], dummy_preds, average="macro")
print(f"Majority-class-only macro-F1: {dummy_f1:.3f}")
print("(If your model's macro-F1 from the training script was only slightly")
print(" above this, the model is barely learning anything from the features.)")

# ---- 2. Mutual information between each feature and y ---------------------
# This does NOT depend on train/test split, model choice, or the graph at
# all. It directly asks: does knowing feature i tell you anything about y?
print("\n=== 2. Per-feature mutual information with label ===")
mi = mutual_info_classif(X, y, random_state=42)
order = np.argsort(mi)[::-1]
for rank, i in enumerate(order[:10]):
    print(f"  feature {i:2d}: MI = {mi[i]:.4f}")
print(f"Sum of all MI: {mi.sum():.4f}  (near 0 = features carry almost no info about y)")

# ---- 3. Cross-validated LR score ignoring the given train/test split -----
# If features are informative but the PROVIDED split is what's broken
# (e.g. x/y misaligned only within one of the masks), a fresh CV split
# should still do fine even though the shipped test_mask performs badly.
print("\n=== 3. Fresh 5-fold CV (ignores provided masks) ===")
cv_scores = cross_val_score(
    LogisticRegression(max_iter=2000), X, y, cv=5, scoring="f1_macro"
)
print(f"CV macro-F1 per fold: {np.round(cv_scores,3)}")
print(f"CV macro-F1 mean: {cv_scores.mean():.3f}")
print("(If this is also ~0.3, the problem is the features/labels themselves,")
print(" not the train/val/test masks. If this is high, e.g. > 0.6, the masks")
print(" or the x/y row alignment for this particular split is the problem.)")

# ---- 4. Shuffle test: is y possibly permuted relative to X? ---------------
# Fit on train, but check accuracy of predicting a *shuffled* copy of y_test
# vs the real y_test. This just confirms the metric/pipeline itself is sane
# and gives a sanity floor for comparison.
print("\n=== 4. Shuffled-label control ===")
rng = np.random.RandomState(42)
y_test_shuffled = y[test_idx].copy()
rng.shuffle(y_test_shuffled)
lr = LogisticRegression(max_iter=2000).fit(X[train_idx], y[train_idx])
preds = lr.predict(X[test_idx])
real_f1 = f1_score(y[test_idx], preds, average="macro")
shuffled_f1 = f1_score(y_test_shuffled, preds, average="macro")
print(f"LR macro-F1 vs real y_test:     {real_f1:.3f}")
print(f"LR macro-F1 vs shuffled y_test: {shuffled_f1:.3f}")
print("(Real should clearly beat shuffled if the model learned anything at all.")
print(" If real and shuffled are close, X and y may be misaligned for this split.)")

print("\n=== How to read this ===")
print("- Sum of MI near 0            -> features don't encode label info at all")
print("                                 (possible x/y misalignment across the WHOLE file,")
print("                                  or features genuinely don't predict this label)")
print("- Sum of MI clearly > 0       -> features DO carry signal")
print("    + fresh CV score is high  -> the shipped train/val/test masks are the problem")
print("    + fresh CV score is also low -> something deeper (check label")
print("      generation logic, or whether X rows got reordered vs y at some point)")

=== 0. Basic integrity checks ===
X shape: (1000, 34), y shape: (1000,)
NaNs in X: 0, NaNs in y: 0
Duplicate rows in X: 0
Class balance: [270 449 281] (proportions: [0.27  0.449 0.281])

=== 1. Trivial floor ===
Majority-class-only macro-F1: 0.206
(If your model's macro-F1 from the training script was only slightly
 above this, the model is barely learning anything from the features.)

=== 2. Per-feature mutual information with label ===
  feature  1: MI = 0.0477
  feature  8: MI = 0.0276
  feature 21: MI = 0.0248
  feature 13: MI = 0.0230
  feature 18: MI = 0.0190
  feature  6: MI = 0.0185
  feature 28: MI = 0.0161
  feature  4: MI = 0.0146
  feature 19: MI = 0.0108
  feature  3: MI = 0.0106
Sum of all MI: 0.2683  (near 0 = features carry almost no info about y)

=== 3. Fresh 5-fold CV (ignores provided masks) ===
CV macro-F1 per fold: [0.309 0.329 0.297 0.258 0.334]
CV macro-F1 mean: 0.305
(If this is also ~0.3, the problem is the features/labels themselves,
 not the train/val/test m

In [5]:
"""
Group7 dropout prediction — full pipeline (v2, redesigned label/features).

Run in Colab:
  1. !pip install torch_geometric -q
  2. Upload your new .pt file to the Colab session
  3. Set DATA_PATH below to match its filename
  4. Run this script top to bottom

This version checks the data BEFORE training any model, since the last
two files both had hidden problems that training alone didn't reveal
clearly until we dug in. If Section A below looks bad, stop and fix the
data before trusting anything from Section B onward.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch_geometric.nn import GATConv, SAGEConv
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score

DATA_PATH = "Real_school_data_full.pt"  # <-- change to your new file's name
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = torch.load(DATA_PATH, weights_only=False).to(device)

print(f"Device: {device}")
print(f"Nodes: {data.num_nodes} | Edges (incl. self-loops): {data.edge_index.shape[1]}")
print(f"Class counts: {torch.bincount(data.y).tolist()}")
print(f"Train/Val/Test sizes: {int(data.train_mask.sum())}/{int(data.val_mask.sum())}/{int(data.test_mask.sum())}")

X = data.x.cpu().numpy()
y = data.y.cpu().numpy()
train_idx = data.train_mask.cpu().numpy()
test_idx = data.test_mask.cpu().numpy()

# =========================================================================
# SECTION A: Does the data actually have a usable signal? (check BEFORE
# trusting any model result below)
# =========================================================================
print("\n" + "=" * 70)
print("SECTION A: DATA SANITY CHECKS")
print("=" * 70)

# A1: trivial floor
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X[train_idx], y[train_idx])
dummy_f1 = f1_score(y[test_idx], dummy.predict(X[test_idx]), average="macro")
print(f"\n[A1] Majority-class-only macro-F1 (trivial floor): {dummy_f1:.3f}")

# A2: mutual information per feature (model-free check)
mi = mutual_info_classif(X, y, random_state=SEED)
order = np.argsort(mi)[::-1]
print(f"\n[A2] Top 10 features by mutual information with label:")
for i in order[:10]:
    print(f"     feature {i:2d}: MI = {mi[i]:.4f}")
print(f"     Sum of all MI: {mi.sum():.4f}")
print("     (near 0 = features barely relate to label; last time this was 0.18 — a red flag)")

# A3: fresh CV, ignoring provided masks
cv_scores = cross_val_score(LogisticRegression(max_iter=2000), X, y, cv=5, scoring="f1_macro")
print(f"\n[A3] Fresh 5-fold CV macro-F1 (ignores provided masks): {cv_scores.mean():.3f}  (folds: {np.round(cv_scores,3)})")

# A4: shuffled-label control
rng = np.random.RandomState(SEED)
y_test_shuffled = y[test_idx].copy()
rng.shuffle(y_test_shuffled)
lr_check = LogisticRegression(max_iter=2000).fit(X[train_idx], y[train_idx])
preds_check = lr_check.predict(X[test_idx])
real_f1 = f1_score(y[test_idx], preds_check, average="macro")
shuffled_f1 = f1_score(y_test_shuffled, preds_check, average="macro")
print(f"\n[A4] LR macro-F1 vs real labels:     {real_f1:.3f}")
print(f"     LR macro-F1 vs shuffled labels: {shuffled_f1:.3f}")
print("     (real should clearly beat shuffled — if not, something is still broken)")

verdict_ok = (mi.sum() > 0.5) and (cv_scores.mean() > dummy_f1 + 0.1) and (real_f1 > shuffled_f1 + 0.05)
print(f"\n[VERDICT] {'Looks OK to proceed — features show real signal above the trivial floor.' if verdict_ok else 'STILL LOOKS BROKEN — check the data before trusting Section B results below.'}")

# =========================================================================
# SECTION B: Model comparison (only meaningful if Section A looks OK)
# =========================================================================
print("\n" + "=" * 70)
print("SECTION B: MODEL COMPARISON")
print("=" * 70 + "\n")

def eval_sklearn_baseline(model, name):
    model.fit(X[train_idx], y[train_idx])
    preds = model.predict(X[test_idx])
    macro_f1 = f1_score(y[test_idx], preds, average="macro")
    per_class = f1_score(y[test_idx], preds, average=None)
    print(f"{name:22s} macro-F1={macro_f1:.3f}  per-class={np.round(per_class,3)}")
    return macro_f1

lr_f1 = eval_sklearn_baseline(LogisticRegression(max_iter=2000), "Logistic Regression")
rf_f1 = eval_sklearn_baseline(RandomForestClassifier(n_estimators=300, random_state=SEED), "Random Forest")

IN_DIM = data.x.shape[1]
HIDDEN = 16
OUT_DIM = int(data.y.max().item()) + 1

class GAT(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout, edge_dim=1)
        self.conv2 = GATConv(hidden * heads, out_dim, heads=1, concat=False, dropout=dropout, edge_dim=1)

    def forward(self, x, edge_index, edge_attr):
        e = edge_attr.unsqueeze(-1)
        x = F.elu(self.conv1(x, edge_index, e))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, e)
        return x

class SAGE(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, out_dim)

    def forward(self, x, edge_index, edge_attr=None):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

class GatedGAT(nn.Module):
    """Learned per-node gate blending GAT's graph representation against
    a projection of the node's own raw features:
        out = gate * h_graph + (1 - gate) * h_own
    """
    def __init__(self, in_dim, hidden, out_dim, heads=4, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_dim, hidden, heads=heads, dropout=dropout, edge_dim=1)
        self.conv2 = GATConv(hidden * heads, hidden, heads=1, concat=False, dropout=dropout, edge_dim=1)
        self.own_proj = nn.Linear(in_dim, hidden)
        self.gate = nn.Linear(in_dim + hidden, 1)
        self.out = nn.Linear(hidden, out_dim)
        self.last_gate_values = None

    def forward(self, x, edge_index, edge_attr):
        e = edge_attr.unsqueeze(-1)
        h_graph = F.elu(self.conv1(x, edge_index, e))
        h_graph = F.dropout(h_graph, p=self.dropout, training=self.training)
        h_graph = self.conv2(h_graph, edge_index, e)
        h_own = self.own_proj(x)
        g = torch.sigmoid(self.gate(torch.cat([x, h_graph], dim=-1)))
        self.last_gate_values = g.detach()
        h = g * h_graph + (1 - g) * h_own
        return self.out(h)

def train_gnn(model, data, epochs=200, lr=0.01, weight_decay=5e-4, patience=30):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_f1, best_state, stale = 0.0, None, 0
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.edge_attr)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index, data.edge_attr)
            val_preds = out[data.val_mask].argmax(dim=1)
            val_f1 = f1_score(data.y[data.val_mask].cpu(), val_preds.cpu(), average="macro")
        if val_f1 > best_val_f1:
            best_val_f1, best_state, stale = val_f1, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            stale += 1
            if stale >= patience:
                break
    model.load_state_dict(best_state)
    return model

def eval_gnn(model, data, name):
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index, data.edge_attr)
        preds = out[data.test_mask].argmax(dim=1).cpu()
        y_true = data.y[data.test_mask].cpu()
        macro_f1 = f1_score(y_true, preds, average="macro")
        per_class = f1_score(y_true, preds, average=None)
    print(f"{name:22s} macro-F1={macro_f1:.3f}  per-class={np.round(per_class,3)}")
    return macro_f1

torch.manual_seed(SEED)
gat = GAT(IN_DIM, HIDDEN, OUT_DIM).to(device)
gat = train_gnn(gat, data)
gat_f1 = eval_gnn(gat, data, "GAT (unmodified)")

torch.manual_seed(SEED)
sage = SAGE(IN_DIM, HIDDEN, OUT_DIM).to(device)
sage = train_gnn(sage, data)
sage_f1 = eval_gnn(sage, data, "GraphSAGE")

torch.manual_seed(SEED)
gated = GatedGAT(IN_DIM, HIDDEN, OUT_DIM).to(device)
gated = train_gnn(gated, data)
gated_f1 = eval_gnn(gated, data, "Gated GAT")

gated.eval()
with torch.no_grad():
    _ = gated(data.x, data.edge_index, data.edge_attr)
    gate_vals = gated.last_gate_values.cpu().numpy().flatten()
print(f"\nGate values -> mean={gate_vals.mean():.3f} std={gate_vals.std():.3f} "
      f"min={gate_vals.min():.3f} max={gate_vals.max():.3f}")
print("(mean close to 0 = leaning on own features; close to 1 = trusting the graph)")

print("\n=== SUMMARY (test set macro-F1) ===")
print(f"Logistic Regression: {lr_f1:.3f}")
print(f"Random Forest:       {rf_f1:.3f}")
print(f"GraphSAGE:           {sage_f1:.3f}")
print(f"GAT (unmodified):    {gat_f1:.3f}")
print(f"Gated GAT:           {gated_f1:.3f}")
print(f"\nDelta (Gated GAT - unmodified GAT): {gated_f1 - gat_f1:+.3f}")

Device: cpu
Nodes: 1000 | Edges (incl. self-loops): 4068
Class counts: [270, 449, 281]
Train/Val/Test sizes: 700/150/150

SECTION A: DATA SANITY CHECKS

[A1] Majority-class-only macro-F1 (trivial floor): 0.206

[A2] Top 10 features by mutual information with label:
     feature  1: MI = 0.0477
     feature  8: MI = 0.0276
     feature 21: MI = 0.0248
     feature 13: MI = 0.0230
     feature 18: MI = 0.0190
     feature  6: MI = 0.0185
     feature 28: MI = 0.0161
     feature  4: MI = 0.0146
     feature 19: MI = 0.0108
     feature  3: MI = 0.0106
     Sum of all MI: 0.2683
     (near 0 = features barely relate to label; last time this was 0.18 — a red flag)

[A3] Fresh 5-fold CV macro-F1 (ignores provided masks): 0.305  (folds: [0.309 0.329 0.297 0.258 0.334])

[A4] LR macro-F1 vs real labels:     0.279
     LR macro-F1 vs shuffled labels: 0.365
     (real should clearly beat shuffled — if not, something is still broken)

[VERDICT] STILL LOOKS BROKEN — check the data before trusting

In [6]:
"""
Group7 dropout prediction — feature-to-feature relationship check.

This does NOT look at the label at all. It only asks: do the features
in this file relate to EACH OTHER in sensible, real-world ways?
(e.g. does household income relate to travel time / mode of transport,
the way it plausibly should for real students?)

Run in Colab (torch already available; no extra installs needed for this one
beyond what a fresh Colab already has, unless torch_geometric isn't loaded
yet -- if so, run: !pip install torch_geometric -q first).
"""

import torch
import numpy as np
import pandas as pd

DATA_PATH = "Real_school_data_full.pt"  # <-- set to your file's name
data = torch.load(DATA_PATH, weights_only=False)

X = data.x.numpy()
n_features = X.shape[1]
print(f"Checking {n_features} features across {X.shape[0]} students\n")

# ---- 1. Full feature-to-feature correlation matrix ------------------------
df = pd.DataFrame(X, columns=[f"f{i}" for i in range(n_features)])
corr = df.corr()

# ---- 2. Strongest pairwise relationships (excluding a feature with itself) -
corr_no_diag = corr.copy()
np.fill_diagonal(corr_no_diag.values, 0)

# get top 15 strongest absolute correlations, each pair counted once
pairs = []
for i in range(n_features):
    for j in range(i + 1, n_features):
        pairs.append((i, j, corr_no_diag.iloc[i, j]))
pairs.sort(key=lambda t: abs(t[2]), reverse=True)

print("=== Top 15 strongest feature-to-feature correlations ===")
for i, j, c in pairs[:15]:
    print(f"  feature {i:2d} <-> feature {j:2d}:  corr = {c:+.3f}")

# ---- 3. Overall summary stat -----------------------------------------------
all_abs_corrs = [abs(c) for _, _, c in pairs]
print(f"\nMean absolute correlation across all feature pairs: {np.mean(all_abs_corrs):.4f}")
print(f"Max absolute correlation (excluding self):            {np.max(all_abs_corrs):.4f}")
print(f"Number of pairs with |corr| > 0.3:                    {sum(1 for c in all_abs_corrs if c > 0.3)}")
print(f"Number of pairs with |corr| > 0.5:                    {sum(1 for c in all_abs_corrs if c > 0.5)}")

print("\n=== How to read this ===")
print("- If several pairs show |corr| > 0.3-0.5, features DO relate to each")
print("  other in a real way -- there's structure for clustering/unsupervised")
print("  methods to find.")
print("- If the top correlations are all weak (say, everything under 0.15-0.2,")
print("  similar to what random noise would produce), the features are close")
print("  to independent of each other too -- meaning unsupervised learning")
print("  would face the same 'nothing to find' problem as supervised did.")
print("\nTip: once you see WHICH feature numbers show the strongest pairs,")
print("check back against your pipeline notebook's `feature_cols_final` list")
print("to see which real-world columns those numbers correspond to -- e.g.")
print("is feature 5 'Household income level' and feature 12 'Travel time'?")
print("If so, does a positive/negative correlation between them make sense")
print("in real life? That's the actual judgment call this check sets up.")

Checking 34 features across 1000 students

=== Top 15 strongest feature-to-feature correlations ===
  feature  4 <-> feature  5:  corr = +0.744
  feature 30 <-> feature 31:  corr = -0.725
  feature 18 <-> feature 19:  corr = -0.691
  feature  2 <-> feature  4:  corr = +0.679
  feature 25 <-> feature 30:  corr = -0.674
  feature 12 <-> feature 13:  corr = -0.659
  feature 14 <-> feature 16:  corr = -0.637
  feature  3 <-> feature 31:  corr = -0.610
  feature 14 <-> feature 15:  corr = -0.551
  feature 25 <-> feature 28:  corr = -0.528
  feature  2 <-> feature  5:  corr = +0.525
  feature 11 <-> feature 12:  corr = -0.512
  feature 25 <-> feature 26:  corr = -0.500
  feature 25 <-> feature 27:  corr = -0.495
  feature 20 <-> feature 21:  corr = -0.494

Mean absolute correlation across all feature pairs: 0.0549
Max absolute correlation (excluding self):            0.7439
Number of pairs with |corr| > 0.3:                    28
Number of pairs with |corr| > 0.5:                    13

=== 

In [7]:
print(list(df[feature_cols_final].columns))

NameError: name 'feature_cols_final' is not defined